In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path("../src").resolve()))
from features import build_model_frame

In [2]:
df = pd.read_csv("../data/raw/shots.csv")
print(f"Raw shape: {df.shape}")
df.head()

Raw shape: (218700, 24)


,grid_type,game_id,game_event_id,player_id,player_name,team_id,team_name,period,minutes_remaining,seconds_remaining,...,shot_zone_area,shot_zone_range,shot_distance,loc_x,loc_y,shot_attempted_flag,shot_made_flag,game_date,htm,vtm
0,Shot Chart Detail,22300018,12,1627749,Dejounte Murray,1610612737,Atlanta Hawks,1,10,56,...,Center(C),Less Than 8 ft.,6,-26,64,1,1,20231114,DET,ATL
1,Shot Chart Detail,22300018,16,1630552,Jalen Johnson,1610612737,Atlanta Hawks,1,10,18,...,Left Side Center(LC),24+ ft.,26,-202,178,1,1,20231114,DET,ATL
2,Shot Chart Detail,22300018,25,203992,Bogdan Bogdanović,1610612737,Atlanta Hawks,1,9,34,...,Left Side(L),8-16 ft.,15,-151,20,1,1,20231114,DET,ATL
3,Shot Chart Detail,22300018,28,1629631,De'Andre Hunter,1610612737,Atlanta Hawks,1,9,2,...,Center(C),Less Than 8 ft.,3,-4,31,1,1,20231114,DET,ATL
4,Shot Chart Detail,22300018,32,1627749,Dejounte Murray,1610612737,Atlanta Hawks,1,8,33,...,Center(C),Less Than 8 ft.,2,28,6,1,1,20231114,DET,ATL


In [3]:
print(df.columns.tolist())

['grid_type', 'game_id', 'game_event_id', 'player_id', 'player_name', 'team_id', 'team_name', 'period', 'minutes_remaining', 'seconds_remaining', 'event_type', 'action_type', 'shot_type', 'shot_zone_basic', 'shot_zone_area', 'shot_zone_range', 'shot_distance', 'loc_x', 'loc_y', 'shot_attempted_flag', 'shot_made_flag', 'game_date', 'htm', 'vtm']


In [4]:
df_model = build_model_frame(df)
print(f"Model frame shape: {df_model.shape}")
df_model.head()

Model frame shape: (218700, 28)


,grid_type,game_id,game_event_id,player_id,player_name,team_id,team_name,period,minutes_remaining,seconds_remaining,...,loc_y,shot_attempted_flag,shot_made_flag,game_date,htm,vtm,shot_angle,game_seconds_remaining,player_zone_fg_pct,shot_value
0,Shot Chart Detail,22300018,12,1627749,Dejounte Murray,1610612737,Atlanta Hawks,1,10,56,...,64,1,1,20231114,DET,ATL,112.109448,656,0.480938,2
1,Shot Chart Detail,22300018,16,1630552,Jalen Johnson,1610612737,Atlanta Hawks,1,10,18,...,178,1,1,20231114,DET,ATL,138.613881,618,0.384058,3
2,Shot Chart Detail,22300018,25,203992,Bogdan Bogdanović,1610612737,Atlanta Hawks,1,9,34,...,20,1,1,20231114,DET,ATL,172.455071,574,0.480769,2
3,Shot Chart Detail,22300018,28,1629631,De'Andre Hunter,1610612737,Atlanta Hawks,1,9,2,...,31,1,1,20231114,DET,ATL,97.352379,542,0.638462,2
4,Shot Chart Detail,22300018,32,1627749,Dejounte Murray,1610612737,Atlanta Hawks,1,8,33,...,6,1,1,20231114,DET,ATL,12.094757,513,0.581169,2


In [5]:
new_cols = set(df_model.columns) - set(df.columns)
print("New columns added:", sorted(new_cols))

New columns added: ['game_seconds_remaining', 'player_zone_fg_pct', 'shot_angle', 'shot_value']


In [6]:
df_model[["loc_x", "loc_y", "shot_distance", "shot_angle"]].describe()

,loc_x,loc_y,shot_distance,shot_angle
count,218700.000000,218700.000000,218700.000000,218700.000000
mean,-1.694572,94.184188,140.130516,79.775708
std,114.291726,94.549932,106.015142,63.504757
min,-250.000000,-51.000000,0.000000,-179.769898
25%,-48.000000,14.000000,31.320920,48.723372
50%,0.000000,53.000000,126.889716,88.210089
75%,45.000000,179.000000,250.571347,122.265709
max,250.000000,842.000000,846.179650,180.000000


In [7]:
df_model[["period", "minutes_remaining", "seconds_remaining", "game_seconds_remaining"]].describe()

,period,minutes_remaining,seconds_remaining,game_seconds_remaining
count,218700.000000,218700.000000,218700.000000,218700.000000
mean,2.482684,5.376644,28.866694,351.465322
std,1.130491,3.451047,17.393807,207.761570
min,1.000000,0.000000,0.000000,0.000000
25%,1.000000,2.000000,14.000000,171.000000
50%,2.000000,5.000000,29.000000,353.000000
75%,3.000000,8.000000,44.000000,532.000000
max,6.000000,11.000000,59.000000,717.000000


In [8]:
df_model[["player_name", "shot_zone_basic", "player_zone_fg_pct"]].head(10)

,player_name,shot_zone_basic,player_zone_fg_pct
0,Dejounte Murray,In The Paint (Non-RA),0.480938
1,Jalen Johnson,Above the Break 3,0.384058
2,Bogdan Bogdanović,Mid-Range,0.480769
3,De'Andre Hunter,Restricted Area,0.638462
4,Dejounte Murray,Restricted Area,0.581169
5,De'Andre Hunter,Restricted Area,0.638462
6,Clint Capela,In The Paint (Non-RA),0.445312
7,De'Andre Hunter,In The Paint (Non-RA),0.449541
8,Dejounte Murray,Restricted Area,0.581169
9,Clint Capela,Restricted Area,0.608247


In [9]:
df_model["shot_value"].value_counts()

shot_value
2    132345
3     86355
Name: count, dtype: int64

In [10]:
model_features = [
    "shot_distance",
    "shot_angle",
    "period",
    "game_seconds_remaining",
    "player_zone_fg_pct",
    "shot_value",
    "shot_made_flag",
]
df_model[model_features].isnull().sum()

shot_distance             0
shot_angle                0
period                    0
game_seconds_remaining    0
player_zone_fg_pct        0
shot_value                0
shot_made_flag            0
dtype: int64

In [11]:
output_path = Path("../data/processed/shots_model_input.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_model.to_csv(output_path, index=False)
print(f"Saved {len(df_model):,} rows to {output_path}")

Saved 218,700 rows to ..\data\processed\shots_model_input.csv


In [12]:
verify = pd.read_csv(output_path)
print(f"Reloaded shape: {verify.shape}")
verify[model_features].head()

Reloaded shape: (218700, 28)


,shot_distance,shot_angle,period,game_seconds_remaining,player_zone_fg_pct,shot_value,shot_made_flag
0,69.079664,112.109448,1,656,0.480938,2,1
1,269.235956,138.613881,1,618,0.384058,3,1
2,152.318745,172.455071,1,574,0.480769,2,1
3,31.256999,97.352379,1,542,0.638462,2,1
4,28.635642,12.094757,1,513,0.581169,2,1
